# Italia Corpus — Struttura e qualità del corpus normativo italiano

**Repo**: [ahmeabd/italia-corpus](https://github.com/ahmeabd/italia-corpus) \
**Fork Lab**: [dataciviclab/italia-corpus](https://github.com/dataciviclab/italia-corpus)

Italia Corpus è un progetto che converte la legislazione italiana (da Normattiva) in file Markdown.
Contiene oltre 280.000 atti normativi, dai regi decreti del Regno d'Italia ai DL del 2026.

Questo notebook analizza:
1. **Struttura del corpus** — quali collezioni ci sono, quanto pesano, cosa è vivo e cosa è storico
2. **Qualità del dataset** `normativa_recepimento_ue` — 1.076 atti di recepimento direttive UE con metadati estratti

---
## 1. Struttura del corpus

L'analisi è fatta dal git tree object (non scarica i file, solo i metadati del repository).
Legge 288.279 file in circa 2 secondi.

In [ ]:
import subprocess
from collections import Counter
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent

out = subprocess.run(
    ['git', 'ls-tree', '-r', 'HEAD'],
    capture_output=True, text=True, timeout=30,
    cwd=REPO
).stdout

files = []
for line in out.strip().split('\n'):
    if not line.strip():
        continue
    meta, path = line.split('\t')
    files.append({'path': path})

print(f'File totali: {len(files):,}')

In [ ]:
# Per tipo di file
exts = Counter()
for f in files:
    p = f['path']
    if '.' in p:
        ext = '.' + p.rsplit('.', 1)[1].lower()
    else:
        ext = '(nessuna)'
    exts[ext] += 1

print(f"{'Estensione':>20} | {'File':>10} | {'%':>6}")
print('-' * 42)
for ext, count in exts.most_common(10):
    print(f'{ext:>20} | {count:>10,} | {count*100/len(files):>5.1f}%')
print(f'\nFile .md: {exts.get(".md", 0):,} ({exts.get(".md", 0)*100/len(files):.1f}%)')
print(f'File .md\" (nome con virgoletta finale): {exts.get(".md\"", 0):,} — da verificare)')

In [ ]:
# Per collezione (directory di primo livello)
dirs = Counter()
for f in files:
    path = f['path']
    d = path.split('/')[0] if '/' in path else '(root)'
    dirs[d] += 1

total = len(files)
print(f"{'Collezione':>55} | {'File':>10} | {'%':>6} | {'Cumulativa':>10}")
print('-' * 86)
cum = 0
for d, count in dirs.most_common(30):
    cum += count
    print(f'{d:>55} | {count:>10,} | {count*100/total:>5.1f}% | {cum*100/total:>6.1f}%')

# Categorie
vive_nomi = ['DL e leggi di conversione', 'Decreti Legislativi', 'Leggi di ratifica',
        'Regolamenti ministeriali', 'Regolamenti governativi',
        'Atti di recepimento direttive UE', 'DPCM',
        'Decreti legislativi luogotenenziali', 'Leggi delega e relativi provvedimenti delegati',
        'Regolamenti di delegificazione', 'Testi Unici', 'Codici',
        'Leggi costituzionali', 'Leggi finanziarie e di bilancio',
        'Leggi contenenti deleghe', 'Atti di attuazione Regolamenti UE',
        'Leggi di delegazione europea']
vivi_file = sum(dirs[d] for d in vive_nomi if d in dirs)

storiche_nomi = ['Atti normativi abrogati (in originale)', 'Regi decreti', 'DPR']
stor_file = sum(dirs[d] for d in storiche_nomi if d in dirs)

print(f'\n---')
print(f'Collezioni di attualità normativa: {vivi_file:,} file ({vivi_file*100/total:.1f}%)')
print(f'Collezioni storiche (abrogati, regi decreti, DPR): {stor_file:,} file ({stor_file*100/total:.1f}%)')

### Cosa significa

Il **90%** del corpus è storia archiviata: atti abrogati e norme pre-repubblicane.
Solo l'**8%** (~24.000 file) copre la produzione normativa corrente (leggi, DL, decreti, regolamenti, recepimento UE).

Per un uso mirato (ricerca, MCP, estrazione dati) ha senso concentrarsi sulle collezioni vive,
riducendo il volume da 475 MB a circa 50-70 MB.

---
## 2. Dataset normativa_recepimento_ue — Controllo qualità

Il dataset è generato da `lab_tools.extract()`: estrae metadati strutturati dagli atti di recepimento delle direttive UE.
Ogni file Markdown nella collezione "Atti di recepimento direttive UE" viene parsato per estrarre:
tipo atto, data, numero, oggetto, entrata in vigore, riferimenti CELEX e ritardo di recepimento.

In [ ]:
import pandas as pd

df = pd.read_parquet(REPO / 'data' / 'derived' / 'normativa_recepimento_ue.parquet')
print(f'Atti analizzati: {len(df)}')
print(f'Colonne: {list(df.columns)}')
print(f'Periodo: {df.anno_atto.min()} - {df.anno_atto.max()}')

In [ ]:
# Completezza
print(f"{'Campo':>25} | {'Compilati':>10} | {'%':>6} | {'Vuoti':>10}")
print('-' * 56)
for col in df.columns:
    if col in ('ritardo',):
        nulls = (df[col].isna()).sum()
    elif col == 'anno_dir':
        nulls = (df[col] == 0).sum()
    elif col in ('celex', 'entrata_vigore'):
        nulls = (df[col] == '').sum()
    else:
        nulls = df[col].isna().sum()
    filled = len(df) - nulls
    print(f'{col:>25} | {filled:>10,} | {filled*100/len(df):>5.1f}% | {nulls:>10,}')

In [ ]:
# Tipologia degli atti
print('Per tipo di atto:')
print(df['tipo'].value_counts().to_string())
print(f'\nI decreti legislativi rappresentano {771/1076*100:.1f}% del totale — sono lo strumento principale di recepimento UE.')

In [ ]:
# Ritardo di recepimento
r = df['ritardo'].dropna()
print(f'Ritardo (anni) tra pubblicazione della direttiva UE e atto di recepimento:')
print(f'  Minimo:     {r.min():.0f}')
print(f'  Massimo:    {r.max():.0f}')
print(f'  Medio:      {r.mean():.1f}')
print(f'  Mediano:    {r.median():.0f}')
print(f'  Oltre 5 anni: { (r > 5).sum() } atti ({ (r > 5).sum()/len(r)*100:.1f}%)')
print()

bins = list(range(0, 26, 2))
print(f'{"Intervallo":>12} | {"Atti":>6}')
print('-' * 22)
for i in range(len(bins)-1):
    cnt = ((r >= bins[i]) & (r < bins[i+1])).sum()
    if cnt:
        bar = '█' * cnt
        print(f'{bins[i]:>3}-{bins[i+1]-1:<3}  | {cnt:>6} {bar}')

In [ ]:
# Copertura CELEX
con_celex = (df.celex != '').sum()
print(f'Atti con CELEX: {con_celex} / {len(df)} ({con_celex/len(df)*100:.1f}%)')
celex_counts = df[df.celex != '']['celex'].str.split(';').apply(len)
print(f'CELEX per atto: media {celex_counts.mean():.1f}, massimo {celex_counts.max()}')
print(f'CELEX totali: {celex_counts.sum():,}')

In [ ]:
# Andamento temporale
anni = df['anno_atto'].value_counts().sort_index()
print('Atti di recepimento per anno:')
print(f'{"Anno":>6} | {"Atti":>6}')
print('-' * 15)
for anno, cnt in anni.items():
    bar = '█' * min(cnt, 40)
    print(f'{anno:>6} | {cnt:>6} {bar}')

In [ ]:
# Entrata in vigore: gap tra pubblicazione e decorrenza
df_vig = df[(df.entrata_vigore != '') & (df.data != '')].copy()
if len(df_vig) > 0:
    df_vig['data_dt'] = pd.to_datetime(df_vig['data'], errors='coerce')
    df_vig['vigore_dt'] = pd.to_datetime(df_vig['entrata_vigore'], errors='coerce')
    df_vig['gap'] = (df_vig['vigore_dt'] - df_vig['data_dt']).dt.days
    gap = df_vig['gap'].dropna()
    print(f'Giorni tra pubblicazione ed entrata in vigore ({len(df_vig)} atti):')
    print(f'  Minimo:  {gap.min():.0f}')
    print(f'  Massimo: {gap.max():.0f}')
    print(f'  Medio:   {gap.mean():.1f}')
    print(f'  Mediano: {gap.median():.0f}')

---
## Osservazioni

### Sul corpus
- **288.279 file**, di cui 283.745 Markdown (98.4%)
- Il **90%** del volume è composto da atti abrogati, regi decreti e DPR — legislazione storica
- Le collezioni di attualità normativa sono circa **23.000 file**, un sottoinsieme gestibile per ricerca e indicizzazione
- 4.523 file presentano un nome con virgoletta finale (`.md"`) — possibile anomalia nella generazione upstream

### Sul dataset recepimento UE
- **1.076 atti** estratti, dal 1969 al 2026
- **71.4%** con riferimenti CELEX (17.178 CELEX totali, media 22 per atto)
- Ritardo medio di recepimento: **4.1 anni**; massimo: 24 anni
- **74 atti** hanno un ritardo superiore a 5 anni (16% del campione)
- L'entrata in vigore avviene in media **45 giorni** dopo la pubblicazione
- I decreti legislativi sono lo strumento principale (71.7%)

### Limiti noti
- Il 57% degli atti non ha `anno_dir` — la regex di estrazione non cattura tutte le formulazioni
- Il 43% non ha `entrata_vigore` estraibile dal testo
- Il dataset copre solo la collezione "Atti di recepimento direttive UE", non l'intero corpus